# Forks: write-audit-publish (Python sync)

Named, durable, isolated graph branches with structural diff and
promote-back semantics. This notebook walks the
**write-audit-publish** loop end to end:

1. Seed primary with a small social graph.
2. Open a fork; derive new edges from a candidate rule.
3. Audit the result via `db.diff_fork_primary` before deciding.
4. Publish via `db.promote_from_fork` with mixed vertex + edge patterns.
5. Drop the fork; primary now has the new state.

Identity is content-addressed UID for vertices and content-derived
edge UID for edges — unrelated forks with overlapping VIDs still
pair correctly, and parallel edges with different property bags
promote independently.


In [ ]:
import uni_db

## 1. Open a database, define schema

In-memory database with a `Person` label and two edge types:
`KNOWS` (the input social graph) and `FRIEND_OF_FRIEND`
(what our hypothesis will derive).


In [ ]:
db = uni_db.Uni.builder().disable_fork_sweeper(True).build()

db.schema().label("Person").property("name", "string").apply()
db.schema().edge_type("KNOWS", ["Person"], ["Person"]).apply()
db.schema().edge_type("FRIEND_OF_FRIEND", ["Person"], ["Person"]).apply()

print("schema applied")

## 2. Seed primary with a small social graph

Four people, three KNOWS edges in a chain:
`alice → bob → carol → dave`.


In [ ]:
primary = db.session()
tx = primary.tx()
tx.execute(
    "CREATE (alice:Person {name: 'alice'}),"
    "       (bob:Person {name: 'bob'}),"
    "       (carol:Person {name: 'carol'}),"
    "       (dave:Person {name: 'dave'}),"
    "       (alice)-[:KNOWS]->(bob),"
    "       (bob)-[:KNOWS]->(carol),"
    "       (carol)-[:KNOWS]->(dave)"
)
tx.commit()
db.flush()

print("seeded primary with 4 vertices + 3 KNOWS edges")

## 3. Stage a hypothesis on a fork

Fork `hypothesis_a` and run a candidate rule: derive
`FRIEND_OF_FRIEND` edges where two people share a common
KNOWS neighbor (2-hop closure). The fork is fully
writable — same `tx.execute` API as primary.


In [ ]:
fork = primary.fork("hypothesis_a").build()
tx = fork.tx()
tx.execute(
    "MATCH (a:Person)-[:KNOWS]->(b:Person)-[:KNOWS]->(c:Person) "
    "WHERE id(a) <> id(c) "
    "CREATE (a)-[:FRIEND_OF_FRIEND]->(c)"
)
tx.commit()
fork.flush()
del fork  # release the session before promote opens a fresh one

print("hypothesis_a: staged 2-hop FRIEND_OF_FRIEND derivation")

## 4. Audit via `diff_fork_primary`

Returns a `ForkDiff` describing what the fork added,
deleted, and changed. Identity is content-addressed UID
for vertices and content-derived edge UID for edges, so
property mutations surface as added+deleted pairs of
distinct UIDs.


In [ ]:
diff = db.diff_fork_primary("hypothesis_a")

print(
    f"diff: +{len(diff.vertices.added)} vertices, "
    f"+{len(diff.edges.added)} edges, "
    f"~{len(diff.vertices.changed)} changed"
)

for e in diff.edges.added:
    short = e.edge_uid[-8:]
    print(f"  + (…{short}) [:{e.edge_type}] (between two Persons)")

## 5. Publish: `promote_from_fork`

Pass both a vertex pattern and an edge pattern in one
call. All inserts run inside a single primary
transaction; vertices promoted by the first pattern are
visible as endpoint candidates to the second pattern via
an in-memory cache.


In [ ]:
report = db.promote_from_fork(
    "hypothesis_a",
    [
        uni_db.PromotePattern.label("Person"),
        uni_db.PromotePattern.edge_type("FRIEND_OF_FRIEND"),
    ],
)

print(
    f"inserted: {report.vertices_inserted} vertices, "
    f"{report.edges_inserted} edges "
    f"(skipped: {report.vertices_skipped_uid_conflict} UID conflicts, "
    f"{report.edges_skipped_duplicate} dup edges, "
    f"{report.edges_skipped_no_endpoint} orphans)"
)

## 6. Drop the fork; primary now has the new edges


In [ ]:
db.drop_fork("hypothesis_a")

rows = primary.query(
    "MATCH (a:Person)-[r]->(b:Person) "
    "RETURN a.name AS src, type(r) AS rel, b.name AS dst "
    "ORDER BY rel, src, dst"
).rows

for row in rows:
    print(f"  ({row.get('src')})-[:{row.get('rel')}]->({row.get('dst')})")

## Wrap-up

- **Isolation** — the fork's writes never touched primary
  until `promote_from_fork` was called.
- **Audit before publish** — `diff_fork_primary` lets
  you see exactly what the rule produced.
- **Atomic publish** — every pattern in
  `promote_from_fork` runs in one primary transaction.
- **Content-addressed identity** — UID dedup; parallel
  edges with different property bags are distinct.

The same surface exists in Rust (`examples/rust/forks.ipynb`)
and against `uni_db.AsyncUni` (`forks_async.ipynb`).
